# Selected Backdoor Suite Training in Colab

- **BadNet**: patch baseline
- **WaNet**: smooth spatial-warp trigger
- **Blended**: distributed transparent trigger
- **SSBA**: invisible sample-specific trigger, using pre-generated `.npy` poison arrays
- **Low Frequency**: cleaner image-only fifth attack

Each completed run saves the full BackdoorBench `record/<run_name>` folder plus a compact `weights_only.pth`.

## 1. Runtime and Repository

In [ ]:
REPO_URL = 'https://github.com/alepelosi/backdoor_finetuning.git'
REPO_DIR = '/content/backdoor_finetuning'

import os
from pathlib import Path

if not Path(REPO_DIR).exists():
    !git clone {REPO_URL} {REPO_DIR}

%cd /content/backdoor_finetuning
!mkdir -p data/cifar10 data/cifar100 data/gtsrb data/tiny record


## 2. Install Dependencies

In [ ]:
!pip -q install pyyaml tqdm pandas matplotlib scipy scikit-learn scikit-image opencv-python pillow kornia imageio tensorboard pytorch-wavelets

import torch
import torchvision
import pytorch_wavelets

print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('pytorch_wavelets: installed')
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 4. Configure Image Experiments


In [ ]:
# Dataset configs available in config/attack/prototype: cifar10, cifar100, gtsrb, tiny
DATASET = 'cifar10'
DATA_ROOT = './data'
DEVICE = 'cuda:0'

EPOCHS = 100
BATCH_SIZE = 128
NUM_WORKERS = 2
AMP = False
RANDOM_SEED = 0
STOP_ON_ERROR = False

DRIVE_OUT_DIR = '/content/drive/MyDrive/backdoor_results/selected_backdoor_suite'

# Exact image suite from the plan. Remove 'lf' or 'bpp' if you only want one fifth attack.
SELECTED_IMAGE_ATTACKS = ['badnet' 'wanet' 'blended' 'ssba' 'lf' 'bpp']
SELECTED_MODELS = ['preactresnet18']

# Optional Drive folder for generated resources not committed to the repo.
# Expected SSBA file names: cifar10_ssba_train_b1.npy and cifar10_ssba_test_b1.npy, etc.
SSBA_RESOURCE_DIR = '/content/drive/MyDrive/backdoor_resources/ssba'

RUNS = [
    {'attack': attack, 'model': model}
    for attack in SELECTED_IMAGE_ATTACKS
    for model in SELECTED_MODELS
]

# Optional: replace RUNS with custom per-run configs if you want special epochs or extra args.
# RUNS = [
#     {'attack': 'badnet', 'model': 'preactresnet18', 'epochs': 100},
#     {'attack': 'wanet', 'model': 'preactresnet18', 'epochs': 100},
#     {'attack': 'blended', 'model': 'vit_b_16', 'epochs': 50},
#     {'attack': 'ssba', 'model': 'preactresnet18', 'epochs': 100},
#     {'attack': 'lf', 'model': 'preactresnet18', 'epochs': 100},
#     {'attack': 'bpp', 'model': 'preactresnet18', 'epochs': 100},
# ]

RUNS


## 5. Image Attack and Model Catalog

SSBA/LC/PoisonInk/TrojanNN are supported by the wrapper, but they require additional generated resources before they can train.

In [ ]:
from pathlib import Path

ATTACKS = {
    'badnet': {
        'script': 'attack/badnet.py',
        'bd_yaml': 'config/attack/badnet/default.yaml',
        'extra_args': ['--patch_mask_path', './resource/badnet/trigger_image.png'],
        'required_paths': ['./resource/badnet/trigger_image.png'],
    },
    'blended': {
        'script': 'attack/blended.py',
        'bd_yaml': 'config/attack/blended/default.yaml',
        'extra_args': [],
        'required_paths': ['./resource/blended/hello_kitty.jpeg'],
    },
    'sig': {
        'script': 'attack/sig.py',
        'bd_yaml': 'config/attack/sig/default.yaml',
        'extra_args': [],
        'required_paths': [],
    },
    'wanet': {
        'script': 'attack/wanet.py',
        'bd_yaml': 'config/attack/wanet/default.yaml',
        'extra_args': [],
        'required_paths': [],
    },
    'lf': {
        'script': 'attack/lf.py',
        'bd_yaml': 'config/attack/lf/default.yaml',
        'extra_args': [],
        'required_paths': [],
    },
    'bpp': {
        'script': 'attack/bpp.py',
        'bd_yaml': 'config/attack/bpp/default.yaml',
        'extra_args': [],
        'required_paths': [],
    },
    'refool': {
        'script': 'attack/refool.py',
        'bd_yaml': 'config/attack/refool/default.yaml',
        'extra_args': [],
        'required_paths': ['./resource/refool/Refool-SelectedReflectionImages/selected_out-images'],
    },
    'ftrojann': {
        'script': 'attack/ftrojann.py',
        'bd_yaml': 'config/attack/ftrojannn/default.yaml',
        'extra_args': [],
        'required_paths': [],
    },
    'inputaware': {
        'script': 'attack/inputaware.py',
        'bd_yaml': 'config/attack/inputaware/default.yaml',
        'extra_args': [],
        'required_paths': [],
    },
    'lira': {
        'script': 'attack/lira.py',
        'bd_yaml': 'config/attack/lira/default.yaml',
        'extra_args': [],
        'required_paths': [],
    },
    'ctrl': {
        'script': 'attack/ctrl.py',
        'bd_yaml': 'config/attack/ctrl/default.yaml',
        'extra_args': [],
        'required_paths': [],
    },
    'ssba': {
        'script': 'attack/ssba.py',
        'bd_yaml': 'config/attack/ssba/default.yaml',
        'extra_args': [],
        'required_paths': [f'./resource/ssba/{DATASET}_ssba_train_b1.npy', f'./resource/ssba/{DATASET}_ssba_test_b1.npy'],
    },
    'lc': {
        'script': 'attack/lc.py',
        'bd_yaml': 'config/attack/lc/default.yaml',
        'extra_args': [],
        'required_paths': [f'./resource/label-consistent/data/preactresnet18_{DATASET}_16_train.npy', f'./resource/label-consistent/data/adv_dataset/{DATASET}_test_v2.npy'],
    },
    'poison_ink': {
        'script': 'attack/poison_ink.py',
        'bd_yaml': 'config/attack/poison_ink/default.yaml',
        'extra_args': [],
        'required_paths': ['./resource/poison_ink/train', './resource/poison_ink/test'],
    },
    'trojannn': {
        'script': 'attack/trojannn.py',
        'bd_yaml': 'config/attack/trojannn/preactresnet18.yaml',
        'extra_args': [],
        'required_paths': [],
    },
}

SUPPORTED_MODELS = [
    'preactresnet18', 'vgg19_bn', 'convnext_tiny', 'vit_b_16',
    'vgg19', 'densenet161', 'mobilenet_v3_large', 'efficientnet_b3',
    'resnet18', 'resnet34', 'resnet50', 'vgg16', 'densenet121',
    'mobilenet_v2', 'efficientnet_b0', 'vit_b_32', 'vit_l_16', 'vit_l_32',
]

print('Attacks:', ', '.join(ATTACKS))
print('Models:', ', '.join(SUPPORTED_MODELS[:8]), '...')


## 6. Helpers


In [ ]:
import csv
import json
import shutil
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

import torch

ROOT = Path('/content/backdoor_finetuning')
RECORD_DIR = ROOT / 'record'
Path(DRIVE_OUT_DIR).mkdir(parents=True, exist_ok=True)

def patch_repo_compatibility():
    trainer = ROOT / 'utils/trainer_cls.py'
    if trainer.exists():
        text = trainer.read_text()
        fixed = text.replace('np.infty', 'np.inf')
        if fixed != text:
            trainer.write_text(fixed)
            print('Patched NumPy compatibility: np.infty -> np.inf')
    else:
        print(f'Warning: {trainer} does not exist yet.')

patch_repo_compatibility()

def ensure_badnet_trigger():
    trigger = ROOT / 'resource/badnet/trigger_image.png'
    if trigger.exists():
        return
    cmd = [
        sys.executable, 'resource/badnet/generate_white_square.py',
        '--image_size', '32',
        '--square_size', '3',
        '--distance_to_right', '0',
        '--distance_to_bottom', '0',
        '--output_path', str(trigger),
    ]
    subprocess.check_call(cmd, cwd=ROOT)

def ensure_ssba_arrays():
    required = [
        ROOT / f'resource/ssba/{DATASET}_ssba_train_b1.npy',
        ROOT / f'resource/ssba/{DATASET}_ssba_test_b1.npy',
    ]
    if all(p.exists() for p in required):
        return

    (ROOT / 'resource/ssba').mkdir(parents=True, exist_ok=True)
    source_dir = Path(SSBA_RESOURCE_DIR)
    search_roots = []
    if source_dir.exists():
        search_roots.append(source_dir)
    drive_root = Path('/content/drive/MyDrive')
    if drive_root.exists() and drive_root not in search_roots:
        search_roots.append(drive_root)

    for target in required:
        if target.exists():
            continue
        found = None
        for root in search_roots:
            exact = root / target.name
            if exact.exists():
                found = exact
                break
            matches = list(root.rglob(target.name))
            if matches:
                found = matches[0]
                break
        if found:
            shutil.copy2(found, target)
            print(f'Copied SSBA resource {found} -> {target}')

    missing = [p for p in required if not p.exists()]
    if missing:
        msg = (
            'SSBA needs pre-generated poison arrays. I checked SSBA_RESOURCE_DIR and searched MyDrive. '
            'Expected file names:'
            + chr(10)
            + chr(10).join('  - ' + p.name for p in missing)
            + chr(10)
            + 'To locate them manually, run: !find /content/drive/MyDrive -name "cifar10_ssba*.npy"'
        )
        raise FileNotFoundError(msg)

def run_name_for(run):
    return run.get('run_name') or f"{run['attack']}_{run['model']}_{DATASET}_seed{RANDOM_SEED}"

def validate_run(run):
    attack = run['attack']
    model = run['model']
    if attack not in ATTACKS:
        raise ValueError(f'Unknown attack {attack}. Choose from {sorted(ATTACKS)}')
    if model not in SUPPORTED_MODELS:
        raise ValueError(f'Unknown model {model}. Choose from SUPPORTED_MODELS or add it there.')
    if DATASET not in {'cifar10', 'cifar100', 'gtsrb', 'tiny'}:
        raise ValueError('DATASET must be one of cifar10, cifar100, gtsrb, tiny for this notebook.')

    if attack == 'badnet':
        ensure_badnet_trigger()
    if attack == 'ssba':
        ensure_ssba_arrays()

    missing = []
    for rel in ATTACKS[attack].get('required_paths', []):
        if not (ROOT / rel).exists():
            missing.append(rel)
    if attack == 'lf':
        pattern = ROOT / f'resource/lowFrequency/{DATASET}_{model}_0_255.npy'
        if not pattern.exists():
            # One upstream CIFAR-100 ConvNeXT file uses 0_225 in this checkout. Let the attack fail only if needed.
            alt = ROOT / f'resource/lowFrequency/{DATASET}_{model}_0_225.npy'
            if not alt.exists():
                missing.append(str(pattern.relative_to(ROOT)))
    if attack == 'trojannn':
        clean_model = ROOT / f'resource/clean_model/{DATASET}_{model}/clean_model.pth'
        if not clean_model.exists():
            missing.append(str(clean_model.relative_to(ROOT)))
    if missing:
        message = 'Missing resources for this run:' + chr(10) + chr(10).join(f'  - {m}' for m in missing)
        raise FileNotFoundError(message)

def build_command(run):
    attack = run['attack']
    cfg = ATTACKS[attack]
    run_name = run_name_for(run)
    cmd = [
        sys.executable, cfg['script'],
        '--yaml_path', f'./config/attack/prototype/{DATASET}.yaml',
        '--bd_yaml_path', cfg['bd_yaml'],
        '--model', run['model'],
        '--dataset', DATASET,
        '--dataset_path', DATA_ROOT,
        '--epochs', str(run.get('epochs', EPOCHS)),
        '--batch_size', str(run.get('batch_size', BATCH_SIZE)),
        '--num_workers', str(run.get('num_workers', NUM_WORKERS)),
        '--device', run.get('device', DEVICE),
        '--random_seed', str(run.get('random_seed', RANDOM_SEED)),
        '--amp', str(run.get('amp', AMP)),
        '--save_folder_name', run_name,
    ]
    cmd.extend(cfg.get('extra_args', []))
    if attack == 'lf':
        default_pattern = ROOT / f'resource/lowFrequency/{DATASET}_{run["model"]}_0_255.npy'
        alt_pattern = ROOT / f'resource/lowFrequency/{DATASET}_{run["model"]}_0_225.npy'
        if not default_pattern.exists() and alt_pattern.exists():
            cmd.extend(['--lowFrequencyPatternPath', './' + str(alt_pattern.relative_to(ROOT))])
    cmd.extend(run.get('extra_args', []))
    return cmd

def stream_subprocess(cmd, log_path):
    with open(log_path, 'w') as log_file:
        process = subprocess.Popen(
            cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        for line in process.stdout:
            print(line, end='')
            log_file.write(line)
        return process.wait()

def export_weights(record_path, run):
    attack_result = record_path / 'attack_result.pt'
    if not attack_result.exists():
        raise FileNotFoundError(f'{attack_result} was not created.')
    try:
        payload = torch.load(attack_result, map_location='cpu', weights_only=False)
    except TypeError:
        payload = torch.load(attack_result, map_location='cpu')
    if isinstance(payload, dict) and 'model' in payload:
        state_dict = payload['model']
        model_name = payload.get('model_name', run['model'])
        num_classes = payload.get('num_classes')
    else:
        state_dict = payload
        model_name = run['model']
        num_classes = None
    out = record_path / 'weights_only.pth'
    torch.save({
        'state_dict': state_dict,
        'model_name': model_name,
        'num_classes': num_classes,
        'dataset': DATASET,
        'attack': run['attack'],
        'run_name': record_path.name,
    }, out)
    return out

def copy_to_drive(record_path):
    destination = Path(DRIVE_OUT_DIR) / record_path.name
    if destination.exists():
        shutil.rmtree(destination)
    shutil.copytree(record_path, destination)
    return destination

def save_summary(rows):
    summary_path = Path(DRIVE_OUT_DIR) / 'multi_attack_summary.csv'
    keys = ['run_name', 'attack', 'model', 'dataset', 'status', 'record_path', 'weights_path', 'drive_path', 'log_path', 'return_code', 'started_at', 'finished_at', 'error']
    with open(summary_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(rows)
    print('Summary saved to', summary_path)
    return summary_path


## 7. Run Training

In [ ]:
summary_rows = []

for run in RUNS:
    run_name = run_name_for(run)
    record_path = RECORD_DIR / run_name
    log_path = RECORD_DIR / f'{run_name}_colab_train.log'
    started_at = datetime.now().isoformat(timespec='seconds')
    row = {
        'run_name': run_name,
        'attack': run['attack'],
        'model': run['model'],
        'dataset': DATASET,
        'status': 'pending',
        'record_path': str(record_path),
        'weights_path': '',
        'drive_path': '',
        'log_path': str(log_path),
        'return_code': '',
        'started_at': started_at,
        'finished_at': '',
        'error': '',
    }

    print()
    print('=' * 90)
    print(f"Starting {run_name}: attack={run['attack']} model={run['model']} dataset={DATASET}")
    print('=' * 90)

    try:
        validate_run(run)
        if record_path.exists():
            raise FileExistsError(f'{record_path} already exists. Change run_name or delete the old run folder.')
        cmd = build_command(run)
        print('Command:', ' '.join(cmd))
        return_code = stream_subprocess(cmd, log_path)
        row['return_code'] = return_code
        if record_path.exists() and log_path.exists():
            in_run_log = record_path / 'colab_train.log'
            shutil.copy2(log_path, in_run_log)
            row['log_path'] = str(in_run_log)
        if return_code != 0:
            raise RuntimeError(f'Training command failed with return code {return_code}. See {log_path}')
        weights_path = export_weights(record_path, run)
        drive_path = copy_to_drive(record_path)
        row.update({
            'status': 'ok',
            'weights_path': str(weights_path),
            'drive_path': str(drive_path),
        })
        print('Saved weights:', weights_path)
        print('Copied run to:', drive_path)
    except Exception as exc:
        row['status'] = 'failed'
        row['error'] = repr(exc)
        print('FAILED:', repr(exc))
        if STOP_ON_ERROR:
            row['finished_at'] = datetime.now().isoformat(timespec='seconds')
            summary_rows.append(row)
            save_summary(summary_rows)
            raise
    finally:
        if record_path.exists() and log_path.exists():
            in_run_log = record_path / 'colab_train.log'
            if str(in_run_log) != row.get('log_path'):
                shutil.copy2(log_path, in_run_log)
                row['log_path'] = str(in_run_log)
        row['finished_at'] = datetime.now().isoformat(timespec='seconds')
        summary_rows.append(row)
        save_summary(summary_rows)

summary_rows


## 8. Inspect Saved Weight

In [ ]:
from pathlib import Path

for row in summary_rows:
    print()
    print(row['run_name'], row['status'])
    drive_path = Path(row['drive_path']) if row['drive_path'] else None
    if drive_path and drive_path.exists():
        for name in ['attack_result.pt', 'weights_only.pth', 'colab_train.log']:
            p = drive_path / name
            print(' ', name, 'exists=' + str(p.exists()), 'size=' + (str(p.stat().st_size) if p.exists() else 'NA'))
    else:
        print(' no Drive copy for this run')


## 9. Download

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

run_name = "ssba_preactresnet18_cifar10_seed0"

src = Path("/content/backdoor_finetuning/record") / run_name
zip_base = Path("/content") / run_name

assert src.exists(), f"Missing folder: {src}"

zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=src.parent, base_dir=src.name)

print("Created:", zip_path)
print("Size MB:", Path(zip_path).stat().st_size / 1024 / 1024)

files.download(zip_path)